# **Error Handling**

## **What's Covered?**
1. Introduction to Error Handling
    - a
2. Defensive Tooling with Schema Validation (Input Layer)
    - The Problem
    - The Solution
3. Schema validation error with `ToolStrategy` (Input Layer)
4. A

## **Introduction to Error Handling**

### **Understanding the Agentic System Design**
1. Decision Layer (LLM) - Control hallucinations and reasoning - typically System Prompt and Structured Output is used to steer the behaviour
2. Schema Layer - Parsing and Validating the LLM Output - Enforces schema (Pydantic)
3. Runtime Layer (Tools) - Control duplicate bookings, partial writes, timeouts, stuck threads, etc... - Enforce a Tool Execution Contract
4. Orchestration Layer

### **Preventing Bad State, Not just Errors**
- LLMs don’t "fail fast", they loop, retry, hallucinate, or degrade silently
- Most cost explosions (like your 429 issue) are not exceptions, they’re bad control flow
- Error handling = catching exceptions at each layer
- In production **Agentic System**: Preventing bad states, not just handling errors
- Retries: Not all failures should be retried 

## **Defensive Tooling with Schema Validation**
**Designing tools** so that they are **physically impossible to invoke with invalid state**, while providing the LLM with clear recovery instructions.

### **The Problem**
Let's look at a sample use case.

In [13]:
from langchain_openai import ChatOpenAI

# Setup API Key
f = open('keys/.openai_api_key.txt')
OPENAI_API_KEY = f.read()

openai_chat_model = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model="gpt-4o-mini",
    temperature=0.0
)

openai_chat_model.invoke("Hi").content

'Hello! How can I assist you today?'

In [18]:
from pydantic import BaseModel, Field
from typing import Literal

class UpdateSubscriptionInput(BaseModel):
    user_id: int = Field(description="The numeric ID of the user.")
    new_plan: str = Field(
        description="The target subscription level."
    )

In [19]:
from langchain_core.tools import tool

@tool(args_schema=UpdateSubscriptionInput)
def update_subscription(user_id: int, new_plan: str):
    """Updates the user's subscription plan."""
    # Because of the Enum, we don't need to check if 'new_plan' is valid inside the function
    return f"Successfully updated user {user_id} to {new_plan}."

In [20]:
from langchain.agents import create_agent

agent = create_agent(
    model=openai_chat_model,
    tools=[update_subscription]
)

In [21]:
response = agent.invoke({"messages" : "My user id is 123, can you update my plan to super-premium-pro"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

My user id is 123, can you update my plan to super-premium-pro
================================== Ai Message ==================================
Tool Calls:
  update_subscription (call_AQYxUljQ3garHP04l3Xmk7GA)
 Call ID: call_AQYxUljQ3garHP04l3Xmk7GA
  Args:
    user_id: 123
    new_plan: super-premium-pro
================================= Tool Message =================================
Name: update_subscription

Successfully updated user 123 to super-premium-pro.
================================== Ai Message ==================================

Your subscription plan has been successfully updated to super-premium-pro.


### **The Solution**

In a real-time system, like a customer support agent that modifies user subscriptions, you cannot afford "garbage in, garbage out." If the agent tries to call update_subscription(user_id=123, plan="super-premium-pro") but that plan name doesn't exist in your database, your tool will crash, potentially leaving the database in an inconsistent state.

In [22]:
from pydantic import BaseModel, Field
from typing import Literal

# Schema with strict validation using Literal
class UpdateSubscriptionInput(BaseModel):
    user_id: int = Field(description="The numeric ID of the user.")
    new_plan: Literal["Basic", "Pro", "Enterprise"] = Field(
        description="The target subscription level. Must be one of: basic, pro, enterprise."
    )

In [23]:
from langchain_core.tools import tool

@tool(args_schema=UpdateSubscriptionInput)
def update_subscription(user_id: int, new_plan: Literal["Basic", "Pro", "Enterprise"]):
    """Updates the user's subscription plan."""
    # Because of the Enum, we don't need to check if 'new_plan' is valid inside the function
    return f"Successfully updated user {user_id} to {new_plan}."

In [24]:
from langchain.agents import create_agent

agent = create_agent(
    model=openai_chat_model,
    tools=[update_subscription]
)

In [25]:
response = agent.invoke({"messages" : "My user id is 123, can you update my plan to super-premium-pro"})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

My user id is 123, can you update my plan to super-premium-pro
================================== Ai Message ==================================

The available subscription plans are Basic, Pro, and Enterprise. "Super-premium-pro" is not a valid option. Please choose one of the available plans.


## **Schema validation error with `ToolStrategy`**

- When structured output doesn’t match the expected schema, the agent provides specific error feedback.
- Feeding errors back into the agent's history helps it perform **"Self-Correction"**.

In [20]:
from pydantic import BaseModel, Field

class ProductRating(BaseModel):
    rating: int | None = Field(description="Rating from 1-5", ge=1, le=5)
    comment: str = Field(description="Review comment")

In [21]:
agent = create_agent(
    model=openai_chat_model,
    tools=[],
    response_format=ToolStrategy(ProductRating),  # Default: handle_errors=True
    handle_errors=True,
    system_prompt="""You are a helpful assistant that parses product reviews and ratings. 
                     Parse EXACTLY as user says. Do NOT fix values. Even if user gives rating >5.
                     Do not make any false field or false value."""
)

In [22]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "User mentioned Amazing product, 10/10!"}]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

User mentioned Amazing product, 10/10!
================================== Ai Message ==================================
Tool Calls:
  ProductRating (call_Fh1C8ZQixfGVDUSie8VDfd7P)
 Call ID: call_Fh1C8ZQixfGVDUSie8VDfd7P
  Args:
    rating: 10
    comment: Amazing product, 10/10!
================================= Tool Message =================================
Name: ProductRating

Error: Failed to parse structured output for tool 'ProductRating': Failed to parse data to ProductRating: 1 validation error for ProductRating
rating
  Input should be less than or equal to 5 [type=less_than_equal, input_value=10, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal.
 Please fix your mistakes.
================================== Ai Message ==================================
Tool Calls:
  ProductRating (call_cQ55XfXrKWhsSGWOsNRuWXTA)
 Call ID: call_cQ55XfXrKWhsSGWOsNRu

### **Important Consideration**

**ISSUE 1:** Some models will intentionally fix 10 to 5 at server-side because they knows your schema constraints (ge=1, le=5) from the description.
> **SOLUTION 1:** Use a strict prompt to enforce model to parse the EXACT values.

**ISSUE 2:** Groq does strict validation server-side. Due to this it will throw 400 error even before LangChain's `handle_error=True` can catch/convert to `ToolMessage` retry. 
> **SOLUTION 2:** We can disable this behaviour by passing `kwargs={"strict": "false"}`.

**Never hide the error:** If you just return "Please try again," the LLM doesn't know why it failed. Always include the string representation of the ValidationError so the LLM knows which field or type was invalid.

**The "3-Strike" Limit:** In production, do not let an agent loop indefinitely on validation errors. Track the number of consecutive ToolMessage errors in your graph State. If it hits a threshold (e.g., 3), force the agent to stop, return a final response to the user, or escalate to a human.